<a href="https://colab.research.google.com/github/abrarhossam51-max/Phantoms-Ai-W4/blob/main/W4D2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, recall_score

# Load the dataset
df = pd.read_csv("train.csv")

# Clean missing values
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

# Drop unnecessary columns
df = df.drop(columns=["Cabin", "Name", "Ticket"])

# Convert categorical columns to numbers
df = pd.get_dummies(
    df,
    columns=["Sex", "Embarked"],
    drop_first=True
)

# Create an imbalanced dataset
class_0 = df[df["Survived"] == 0]
class_1 = df[df["Survived"] == 1].sample(frac=0.1, random_state=42)

df_imbalanced = pd.concat([class_0, class_1])
df_imbalanced = df_imbalanced.sample(frac=1, random_state=42)

print("Class distribution:")
print(df_imbalanced["Survived"].value_counts())

# Separate Features and Label
X = df_imbalanced.drop(columns=["Survived"])
y = df_imbalanced["Survived"]

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Train normal Logistic Regression
model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy_before = accuracy_score(y_test, y_pred)
cm_before = confusion_matrix(y_test, y_pred)
recall_before = recall_score(y_test, y_pred, zero_division=0)

print("\nBefore balancing:")
print("Accuracy:", accuracy_before)
print("Confusion Matrix:")
print(cm_before)
print("Recall:", recall_before)

# Train Logistic Regression with balanced class weights
balanced_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

balanced_model.fit(X_train, y_train)

y_pred_balanced = balanced_model.predict(X_test)

accuracy_after = accuracy_score(y_test, y_pred_balanced)
cm_after = confusion_matrix(y_test, y_pred_balanced)
recall_after = recall_score(
    y_test,
    y_pred_balanced,
    zero_division=0
)

print("\nAfter balancing:")
print("Accuracy:", accuracy_after)
print("Confusion Matrix:")
print(cm_after)
print("Recall:", recall_after)

# Comparison
print("\nComparison:")
print("Recall before balancing:", recall_before)
print("Recall after balancing:", recall_after)

# Explanation:
# Accuracy can be misleading when the dataset is imbalanced.
# A model can get high accuracy by mostly predicting the majority class.
#
# class_weight="balanced" gives more importance to the minority class.
# We compare Recall before and after balancing to see whether
# the model became better at detecting the minority class.